In [ ]:
#setup for processed data
#Note: use for BinaryArray data produced from Entrainment_Preprocessing.ipynb
def GetProcessedString(PROCESSING=False):
    if PROCESSING==True:
        Processed_string="PROCESSED_"
    else:
        Processed_string=""
    return Processed_string

PROCESSING=False 
PROCESSING=True #set to True if using Turbulence-Removed Binary Arrays
Processed_string = GetProcessedString(PROCESSING=PROCESSING)

In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#Importing Libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr

import sys; import os; import time; from datetime import timedelta
import pickle
import h5py
from tqdm import tqdm

In [ ]:
#MAIN DIRECTORIES
def GetDirectories():
    mainDirectory='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
    mainCodeDirectory=os.path.join(mainDirectory,"Code/CodeFiles/")
    scratchDirectory='/mnt/lustre/koa/scratch/air673/'
    codeDirectory=os.getcwd()
    return mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory

[mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory] = GetDirectories()

In [ ]:
#IMPORT CLASSES (from current directory)
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, SlurmJobArray_Class, DataManager_Class

In [ ]:
#IMPORT FUNCTIONS (from current directory)
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from FUNCTIONS_Variable_Calculation import *

In [ ]:
####################################
#LOADING CLASSES

In [ ]:
#data loading class
ModelData = ModelData_Class(mainDirectory, scratchDirectory, simulationNumber=1)
#data manager class
DataManager = DataManager_Class(mainDirectory, scratchDirectory, ModelData.res, ModelData.t_res, ModelData.Nz_str,
                                ModelData.Np_str, dataType="EntrainmentCalculation", dataName="EntrainmentCalculation_DivideMassFlux",
                                dtype='int32')

In [ ]:
#JOB ARRAY SETUP
UsingJobArray=True

def GetNumJobs(res,t_res):
    if res=='1km':
        if t_res=='5min':
            num_jobs=20
        elif t_res=='1min':
            num_jobs=100
    elif res=='250m': 
        if t_res=='1min':
            num_jobs=500
    return num_jobs
num_jobs = GetNumJobs(ModelData.res,ModelData.t_res)
SlurmJobArray = SlurmJobArray_Class(total_elements=ModelData.Ntime, num_jobs=num_jobs, UsingJobArray=UsingJobArray)
start_job = SlurmJobArray.start_job; end_job = SlurmJobArray.end_job

def GetNumElements():
    loop_elements = np.arange(ModelData.Ntime)[start_job:end_job]
    return loop_elements
loop_elements = GetNumElements()

In [ ]:
####################################
#FUNCTIONS

In [ ]:
#CALCULATING ENTRAINMENT
def GetVariable_T(t, varName, cache=None): #*MassFlux
    """Load variable for a given timestep, using cached open HDF5 files if available."""
    if cache is None:
        cache = {}

    timeString = ModelData.timeStrings[t]

    # If file for this timestep already in cache, reuse it
    if timeString in cache:
        f = cache[timeString]
    else:
        # Otherwise, open new file and cache it
        f = DataManager.GetTimestepData_V2(DataManager.inputDataDirectory, timeString)
        cache[timeString] = f

    # Load the desired variable from the open file (lazy read)
    if varName in f.keys():
        output = f[varName][:]  # read actual data
    else:
        # fallback for derived variables
        output = CallVariable(ModelData, DataManager, timeString, variableName=varName)
    return output

def SafeDivide(numerator, denominator):
    # 1. Create an array of NaNs with the same shape as the numerator
    result = np.full(numerator.shape, np.nan, dtype=np.float32)
    
    # 2. Find where we can safely divide
    nonzero = (denominator != 0)
    
    # 3. Only fill the "nonzero" indices with the division result
    result[nonzero] = numerator[nonzero] / denominator[nonzero]
    
    return result
    
# def LoadMassFlux(t): #*MassFlux
#     """
#     lagrangian version
#     """
    
#     updraftType = "g" 
#     massFluxVarName = f"{Processed_string}MassFlux_{updraftType}"
#     MassFlux_g = GetVariable_T(t=t, varName=massFluxVarName, cache=None)

#     updraftType = "c" 
#     massFluxVarName = f"{Processed_string}MassFlux_{updraftType}"
#     MassFlux_c = GetVariable_T(t=t, varName=massFluxVarName, cache=None)

#     return MassFlux_g, MassFlux_c

def LoadMassFlux(t):
    """
    eulerian version
    """
    
    timeString=ModelData.timeStrings[t]

    varName='rho'
    rho = CallVariable(ModelData, DataManager, timeString, 
                                                  variableName=varName)
    
    varName='winterp'
    w = CallVariable(ModelData, DataManager, timeString, 
                                                  variableName=varName)
    
    varName='A_g'
    A_g = CallVariable(ModelData, DataManager, timeString, 
                                                  variableName=varName)
    varName='A_c'
    A_c = CallVariable(ModelData, DataManager, timeString, 
                                                  variableName=varName)
    
    
    MassFlux_g = rho*(A_g*1)
    MassFlux_c = rho*(A_c*1)
    return MassFlux_g,MassFlux_c

def DivideByMassFlux(t, varName,
                     MassFlux_g, MassFlux_c):
    variable = GetVariable_T(t, varName)
    MassFlux = MassFlux_g if "g" in varName else MassFlux_c
    variable_DivideMassFlux = SafeDivide(variable, MassFlux)
    return variable_DivideMassFlux

In [ ]:
def GetVarNames_Entrainment(Processed_string):
    varNames = [f"{Processed_string}Entrainment_g",
                f"{Processed_string}Entrainment_c"]
    
    varNames += [f"{Processed_string}TransferEntrainment_g",
                f"{Processed_string}TransferEntrainment_c"]
    return varNames

def GetVarNames_Detrainment(Processed_string):
    varNames = [f"{Processed_string}Detrainment_g",
                f"{Processed_string}Detrainment_c"]
    
    varNames += [f"{Processed_string}TransferDetrainment_g",
                f"{Processed_string}TransferDetrainment_c"]
    return varNames

def AddDivideMassFluxString(varName):
    parts = varName.rsplit('_', 1)
    varName_DivideMassFlux = f"{parts[0]}_DivideMassFlux_{parts[1]}"
    return varName_DivideMassFlux

In [ ]:
def RunCalculation_Entrainment(t, Processed_string,
                               MassFlux_g, MassFlux_c): 

    varNames = GetVarNames_Entrainment(Processed_string)

    outputDictionary_Variable_DivideMassFlux = {}
    for varName in varNames:
        variable_DivideMassFlux = DivideByMassFlux(t, varName,
                                                   MassFlux_g, MassFlux_c)

        varName_DivideMassFlux = AddDivideMassFluxString(varName)
        outputDictionary_Variable_DivideMassFlux[varName_DivideMassFlux] = variable_DivideMassFlux
        
    return outputDictionary_Variable_DivideMassFlux

def RunCalculation_Detrainment(t, Processed_string,
                               MassFlux_g, MassFlux_c): 

    varNames = GetVarNames_Detrainment(Processed_string)

    outputDictionary_Variable_DivideMassFlux = {}
    for varName in varNames:
        variable_DivideMassFlux = DivideByMassFlux(t, varName,
                                                   MassFlux_g, MassFlux_c)

        varName_DivideMassFlux = AddDivideMassFluxString(varName)
        outputDictionary_Variable_DivideMassFlux[varName_DivideMassFlux] = variable_DivideMassFlux
        
    return outputDictionary_Variable_DivideMassFlux

In [ ]:
##############################################
#RUNNING

In [ ]:
#running calculation
for t in tqdm(loop_elements, total=len(loop_elements)):
    # if np.mod(t,1)==0: print(f'Current time {t}')
    if t == ModelData.Ntime-1:
        continue

    #loading MassFlux
    [MassFlux_g, MassFlux_c] =  LoadMassFlux(t)

    #calculating variables
    outputDictionary_Entrainment_DivideMassFlux = RunCalculation_Entrainment(t, Processed_string,
                                                                         MassFlux_g, MassFlux_c)
    outputDictionary_Detrainment_DivideMassFlux = RunCalculation_Detrainment(t, Processed_string,
                                                                             MassFlux_g, MassFlux_c)
    
    #outputting
    timeString = ModelData.timeStrings[t]
    
    DataManager.SaveOutputTimestep(DataManager.outputDataDirectory, timeString, outputDictionary_Entrainment_DivideMassFlux, dataName=f"{Processed_string}Entrainment_DivideMassFlux",dtype='float32')

    DataManager.SaveOutputTimestep(DataManager.outputDataDirectory, timeString, outputDictionary_Detrainment_DivideMassFlux, dataName=f"{Processed_string}Detrainment_DivideMassFlux",dtype='float32')

In [ ]:
########################
#TESTING

In [ ]:
# #IMPORT FUNCTIONS

# import sys
# path=os.path.join(mainCodeDirectory,'Functions/')
# sys.path.append(path)

# import NumericalFunctions
# from NumericalFunctions import * # import NumericalFunctions 
# import PlottingFunctions
# from PlottingFunctions import * # import PlottingFunctions

# # # Get all functions in NumericalFunctions
# # import inspect
# # functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# # functions

# #FUNCTIONS

# #APPLYING ENTRAINMENT CONSTANT
# entrainmentConstant = DataManager.LoadCalculations(
#     DataManager.outputDataDirectory.replace('EntrainmentCalculation_DivideMassFlux','EntrainmentCalculation'),
#     dataName="EntrainmentConstant",
#     verbose=False,
# )["entrainmentConstant"]
# def ApplyEntrainmentConstant(variable):
#     """
#     Multiply each array in the input dictionary by the 1D entrainment constant profile.
#     Returns the processed arrays in the same order as the input dictionary.
#     """
    
#     # Return arrays in the same order as input
#     return variable*entrainmentConstant[:, np.newaxis, np.newaxis]

# #APPLYING MASS FLUX CONSTANT
# massFluxConstant = DataManager.LoadCalculations(
#     os.path.join(os.path.dirname(DataManager.outputDataDirectory), "MassFluxCalculation"),
#     dataName="MassFluxConstant",
#     verbose=False,
# )["massFluxConstant"]
# def ApplyMassFluxConstant(variable): #*MassFlux
#     """
#     Multiply each array in the input dictionary by the 1D mass flux constant profile.
#     Returns the processed arrays in the same order as the input dictionary.
#     """

#     # Return arrays in the same order as input
#     return variable * massFluxConstant[:, np.newaxis, np.newaxis]

# def GetData_singletime(t):
    
#     varName = f"{Processed_string}Entrainment_c"
    
#     variable = GetVariable_T(t, varName)
#     [MassFlux_g, MassFlux_c] = LoadMassFlux(t)
#     MassFlux = MassFlux_g if "g" in varName else MassFlux_c
#     MassFlux_eulerian = CalculateMassFlux_eulerian(t)

#     variable=ApplyEntrainmentConstant(variable)
#     MassFlux=ApplyMassFluxConstant(MassFlux)
#     return variable,MassFlux,MassFlux_eulerian

# def GetData_multipletimes(ts):

#     varName = f"{Processed_string}Entrainment_c"
    
#     # Get shape from a sample to pre-allocate
#     sample = GetVariable_T(ts[0], varName)
#     shape = (len(ts), *sample.shape)
    
#     # Pre-allocate
#     variable_array = np.full(shape, np.nan)
#     MassFlux_array = np.full(shape, np.nan)
#     MassFlux_eulerian_array = np.full(shape, np.nan)

#     for i, t in tqdm(enumerate(ts), total=len(ts)):
#         v = GetVariable_T(t, varName)
        
#         mg, mc = LoadMassFlux(t)
#         mf = mg if "g" in varName else mc
#         mfe = CalculateMassFlux_eulerian(t)

#         v=ApplyEntrainmentConstant(v)
#         mf=ApplyMassFluxConstant(mf)
        
#         variable_array[i] = v
#         MassFlux_array[i] = mf
#         MassFlux_eulerian_array[i] = mfe

#     return variable_array, MassFlux_array, MassFlux_eulerian_array

# def CalculateMassFlux_eulerian(t):
    
#     timeString=ModelData.timeStrings[t]

#     varName='rho'
#     rho = CallVariable(ModelData, DataManager, timeString, 
#                                                   variableName=varName)
    
#     varName='winterp'
#     w = CallVariable(ModelData, DataManager, timeString, 
#                                                   variableName=varName)
#     varName='A_c'
#     A_c = CallVariable(ModelData, DataManager, timeString, 
#                                                   variableName=varName)
    
#     MassFlux_eulerian = rho*(A_c*1)
#     return MassFlux_eulerian

# # def TestMassFluxCalculation(ts=np.arange(90,130)):
    
# #     a = CalculateMassFlux_eulerian(ts[0])
# #     for t in tqdm(ts[1:]):
# #         temp=CalculateMassFlux_eulerian(t)
# #         a+=temp
# #         plt.plot(np.mean(temp,axis=(1,2)),ModelData.zh)
# #     b = a/len(ts)
# #     c=np.mean(b,axis=(1,2))
# #     plt.plot(c,ModelData.zh)
    
# def MakePlot(variable, MassFlux, MassFlux_eulerian=None, 
#                       method="lagrangian", type="singletime"):
#     # Plotting - Standardized to 1x4
#     fig, (ax1, ax2, ax3, ax4) = plt.subplots(1, 4, figsize=(18, 5), sharey=True)
#     maxZ = 14
    
#     # Data Processing axis selection
#     if type == "singletime":
#         axis = (1, 2)
#     elif type == "multipletimes":
#         axis = (0, 2, 3)

#     # Core Lagrangian variables (a, b, c)
#     a = np.nanmean(variable, axis=axis)
#     b = np.nanmean(MassFlux, axis=axis)
    
#     # Height Masking
#     mask = ModelData.zh > maxZ
#     a[mask], b[mask], c[mask] = np.nan, np.nan, np.nan

#     if method == "lagrangian":
#         # Lagrangian variables (c)
#         c = np.nanmean(SafeDivide(variable, MassFlux), axis=axis)
        
#         # Subplot 1: Means
#         ax1.plot(a, ModelData.zh, color='blue')
#         ax1.set_title('Entrainment '+r"$(kg / m^3 / s^{-1})$")
#         ax1.set_ylabel('z (km)')

#         ax2.plot(b, ModelData.zh, color='orange')
#         ax2.set_title('MassFlux '+r"$(kg / m^3)$")
        
#         # Subplot 2: Empty or redundant in 1x4 if only 3 plots needed? 
#         # I will keep the Ratio of Means here to match the 4-column aesthetic
#         ax3.plot(SafeDivide(a, b), ModelData.zh, color='blue')
#         ax3.set_title("Ratio of Means")
        
#         # Subplot 4: Mean of Ratio
#         ax4.plot(c, ModelData.zh, color='blue')
#         ax4.set_title("Mean of Ratio")
        
#     elif method == "eulerian":
#         # Eulerian variables (d, e)
#         d = np.nanmean(MassFlux_eulerian, axis=axis)
#         e = np.nanmean(SafeDivide(variable, MassFlux_eulerian), axis=axis)
#         d[mask], e[mask] = np.nan, np.nan

#         # Subplot 1: Entrainment
#         ax1.plot(a, ModelData.zh, color='blue')
#         ax1.set_title('Entrainment '+r"$(kg / m^3 / s^{-1})$")
#         ax1.set_title("Lagrangian Entraiment")
#         ax1.set_ylabel('z (km)')
#         ax1.legend()

#         # Subplot 2: MassFlux (Eulerian)
#         ax2.plot(d, ModelData.zh, color='orange')
#         ax2.set_title('MassFlux '+r"$(kg / m^3)$")
#         ax2.set_title("MassFlux Means")
        
#         # Subplot 3: Ratio of Means
#         ax3.plot(SafeDivide(a, d), ModelData.zh, color='blue')
#         ax3.set_title("Ratio of Means")
        
#         # Subplot 4: Mean of Ratio
#         ax4.plot(e, ModelData.zh, color='blue')
#         ax4.set_title("Mean of Ratio")

#     # Plot features applied to common axes
#     for ax in [ax3, ax4]:
#         ax.set_xlabel(r"$(s^{-1})$")
        
#     plt.ylim(0, maxZ)
#     apply_scientific_notation([ax1, ax2, ax3, ax4])
#     plt.tight_layout()

#     return fig

In [ ]:
# #ONE TIMESTEP ONLY

# # Calculating
# step = 1 if ModelData.t_res == "5min" else 5
# t = 120*step
# variable,MassFlux,MassFlux_eulerian = GetData_singletime(t)

In [ ]:
# #Plotting
# fig=MakePlot(variable,MassFlux,
#              type="singletime",method="lagrangian")

# fig.suptitle(
#     f'Entrainment Divided by MassFlux (lagrangian) Calculation - single time', 
#     fontsize=16, 
#     fontweight='bold', 
#     y=1.05
# )

In [ ]:
# #Plotting
# fig=MakePlot(variable,MassFlux,MassFlux_eulerian,
#              type="singletime",method="eulerian")

# fig.suptitle(
#     f'Entrainment Divided by MassFlux (eulerian) Calculation - single time', 
#     fontsize=16, 
#     fontweight='bold', 
#     y=1.05
# )

In [ ]:
# #MULTIPLE TIMESTEPS

# # # Calculating
# step = 1 if ModelData.t_res == "5min" else 5
# ts = range(90*step, 130*step)
# variable,MassFlux,MassFlux_eulerian = GetData_multipletimes(ts)

In [ ]:
# #Plotting
# fig=MakePlot(variable,MassFlux,
#              type="multipletimes",method='lagrangian')
# fig.suptitle(
#     f'Entrainment Divided by MassFlux (lagrangian) Calculation - averaged over {len(ts)*ModelData.dt/60:.0f} minutes', 
#     fontsize=16, 
#     fontweight='bold', 
#     y=1.05
# )

In [ ]:
# #Plotting
# fig=MakePlot(variable,MassFlux,MassFlux_eulerian,
#              type="multipletimes",method='eulerian')
# fig.suptitle(
#     f'Entrainment Divided by MassFlux (eulerian) Calculation - averaged over {len(ts)*ModelData.dt/60:.0f} minutes', 
#     fontsize=16, 
#     fontweight='bold', 
#     y=1.05
# )